# P1_SOILNET_IMAGENET_LI_NO_SSL_BESTREG

**Scientific question:** What is the contribution of VICReg when P0-v4 LI and supervised factors are fixed?  
**Configuration:** `config/experiments/P1_no_ssl_bestreg.yaml`  
**Dataset/split:** locked P0-v4 manifest and 1,407 train / 289 validation / 231 sealed-test metadata  
**Initialization:** exact hash-locked pre-VicReg ImageNet initialization snapshot; no VICReg checkpoint  
**Checkpoint selection:** strict minimum of `(SM0_RMSE + SM20_RMSE) / 2`; classification is secondary  
**Expected outputs:** `validation_best_regression.pth`, `epoch_60_final.pth`, history, best-checkpoint validation predictions/metrics, metadata, and both SHA256 values.

This notebook is designed for a clean-kernel **Run All**. It automatically
trains only after every preflight assertion passes. It never constructs a
test loader, computes test metrics, or invokes notebook 09.


## 1. Reproducibility and imports

Set the deterministic CUDA workspace before importing Torch. Use only the existing environment and repository modules.


In [ ]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

from pathlib import Path
import json, sys
import torch

REPO = Path.cwd().resolve()
while REPO != REPO.parent and not (REPO / "pyproject.toml").is_file():
    REPO = REPO.parent
if not (REPO / "pyproject.toml").is_file():
    raise RuntimeError("Open this notebook from inside the SoilNet checkout")
sys.path.insert(0, str(REPO / "src"))

from soilnet.training import run_one_batch_preflight, train_experiment
from soilnet.utils import load_experiment_context, print_environment


## 2. Locked configuration and paths

Resolve ignored machine-local roots, recompute all locked hashes, and reject protocol drift before any model/data preflight.


In [ ]:
CONFIG_PATH = REPO / "config/experiments/P1_no_ssl_bestreg.yaml"
EXPECTED_EXPERIMENT_ID = "P1_SOILNET_IMAGENET_LI_NO_SSL_BESTREG"
LOCKED_SPLIT_SHA256 = "8927b8223b8c4c234d264ad9ea62ac2df6161124a79787a2e71eeb5cd23eae2f"
LOCKED_MANIFEST_SHA256 = "8ff45054d4b8e3df9758d0c112dc16719572b2267906fb3a5ed5b3262a6732bd"
LOCKED_VICREG_SHA256 = "42599ac025b8d8c8c5b8cca36624665f155e0ff588cd90de2fd47b08cd2d60d8"

context = load_experiment_context(CONFIG_PATH)
assert context.config["experiment_id"] == EXPECTED_EXPERIMENT_ID
assert context.config["ablation_type"] == "remove_vicreg_ssl_stage"
assert context.manifest_sha256 == LOCKED_MANIFEST_SHA256
assert context.split_sha256 == LOCKED_SPLIT_SHA256
assert context.config["split_counts"] == {"train": 1407, "validation": 289, "test": 231}
assert context.config["epochs"] == 60
assert context.config["batch_size"] == 32
assert context.config["optimizer"] == "Adam"
assert context.config["learning_rate"] == 1e-4
assert context.config["weight_decay"] == 0.0
assert context.config["seed"] == 20260905
assert context.config["checkpoint_selection"] == "minimum_mean_validation_RMSE"
assert context.config["selection_formula"] == "(SM0_RMSE + SM20_RMSE) / 2"
assert context.config["primary_checkpoint"] == "validation_best_regression.pth"
assert context.config["epoch_60_checkpoint"] == "epoch_60_final.pth"
assert context.config["test_evaluation"] == "prohibited_in_training_notebook"
completed_metadata = context.run_dir / "run_metadata.json"
if completed_metadata.is_file() and json.loads(completed_metadata.read_text(encoding="utf-8")).get("training_completed") is True:
    raise RuntimeError(f"STOP: completed run already exists at {context.run_dir}; Run All will not overwrite it")
print({"CONFIG_AND_PATHS": "PASS", "experiment_id": EXPECTED_EXPERIMENT_ID, "run_dir": str(context.run_dir)})


## 3. Environment and CUDA gate

Require CUDA without changing or rebuilding the environment.


In [ ]:
environment = print_environment(context)
if not torch.cuda.is_available():
    raise RuntimeError("GPU_BLOCKED: P1 full run requires CUDA; no CPU fallback is authorized")
device = torch.device("cuda")
print({
    "torch_version": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "gpu_name": torch.cuda.get_device_name(0),
    "device": str(device),
    "CUBLAS_WORKSPACE_CONFIG": os.environ["CUBLAS_WORKSPACE_CONFIG"],
})


## 4. One-batch no-step preflight

Build only train and validation loaders. Use a temporary model for one train forward/backward and one validation forward; never call `optimizer.step`. Any failed assertion stops Run All before training.


In [ ]:
PREFLIGHT_PASSED = False
preflight = run_one_batch_preflight(context, device=device)
assert preflight["status"] == "PASS"
assert preflight["train_samples"] == 1407
assert preflight["validation_samples"] == 289
assert preflight["optimizer"] == "Adam"
assert preflight["optimizer_step_performed"] is False
assert preflight["temporary_model"] is True
assert preflight["research_metrics_created"] is False
assert preflight["test_loader_instantiated"] is False
assert all(preflight["loss_components_finite"].values())
assert context.config["use_li"] is True
assert context.config["ssl_checkpoint"] is None
assert preflight["vicreg_checkpoint_loaded"] is False
assert preflight["load_report"]["source_kind"] == "historical_pre_vicreg_imagenet_snapshot"
assert preflight["load_report"]["checkpoint_sha256"] == context.config["initialization_checkpoint"]["sha256"]
ablation_confirmation = "VICREG_NOT_LOADED; exact hash-locked pre-VICReg ImageNet snapshot loaded"
print({
    "PREFLIGHT": "PASS",
    "experiment_id": EXPECTED_EXPERIMENT_ID,
    "ablation_type": context.config["ablation_type"],
    "device": str(device),
    "GPU_name": torch.cuda.get_device_name(0),
    "train_samples": preflight["train_samples"],
    "validation_samples": preflight["validation_samples"],
    "split_SHA256": context.split_sha256,
    "model_input_shapes": {"image": preflight["train_image_shape"], "LI_tensor": preflight["train_li_shape"]},
    "output_shapes": preflight["output_shapes"],
    "optimizer": preflight["optimizer"],
    "lr": context.config["learning_rate"],
    "batch_size": context.config["batch_size"],
    "epochs": context.config["epochs"],
    "seed": context.config["seed"],
    "checkpoint_init_provenance": context.config["initialization_provenance"],
    "ablation_confirmation": ablation_confirmation,
    "optimizer_step_performed": False,
    "temporary_model": True,
    "research_metrics_created": False,
    "test_loader_instantiated": False,
    "test_evaluated": "NO",
})
PREFLIGHT_PASSED = True


## 5. Automatic 60-epoch BESTREG run

Run All reaches this cell only after preflight succeeds. Training remains config-locked, has no early stopping, retains both checkpoints, reloads the best checkpoint, and exports final validation artifacts from it.


In [ ]:
RUN_TRAINING = True
if PREFLIGHT_PASSED is not True:
    raise RuntimeError("STOP: training cannot start because preflight did not pass")
if RUN_TRAINING is not True:
    raise RuntimeError("STOP: Run All training gate is not enabled")
run_metadata = train_experiment(context, resume_if_available=True)


## 6. Artifact and provenance assertions

Confirm that the completed engine output contains both checkpoints, both hashes, BESTREG history, and best-checkpoint validation provenance.


In [ ]:
expected_artifacts = [
    "validation_best_regression.pth", "epoch_60_final.pth",
    "training_history.csv", "validation_metrics.json",
    "validation_predictions.csv", "run_metadata.json", "checkpoint_sha256.txt",
]
missing = [name for name in expected_artifacts if not (context.run_dir / name).is_file()]
if missing:
    raise RuntimeError(f"STOP: completed run is missing artifacts: {missing}")
assert run_metadata["training_completed"] is True
assert run_metadata["test_evaluated"] == "NO"
assert run_metadata["primary_checkpoint_path"].endswith("validation_best_regression.pth")
assert run_metadata["epoch_60_checkpoint_path"].endswith("epoch_60_final.pth")
assert run_metadata["validation_metrics"]["generated_from_checkpoint_epoch"] == run_metadata["best_epoch"]
print({"ARTIFACTS_AND_PROVENANCE": "PASS", **{name: True for name in expected_artifacts}})


## 7. Final experiment summary

Report train/validation evidence only. The sealed test remains untouched.


In [ ]:
print({
    "experiment_id": run_metadata["experiment_id"],
    "training_completed": run_metadata["training_completed"],
    "ablation_type": run_metadata["ablation_type"],
    "primary_checkpoint_path": run_metadata["primary_checkpoint_path"],
    "primary_checkpoint_sha256": run_metadata["primary_checkpoint_sha256"],
    "epoch_60_checkpoint_path": run_metadata["epoch_60_checkpoint_path"],
    "epoch_60_checkpoint_sha256": run_metadata["epoch_60_checkpoint_sha256"],
    "best_epoch": run_metadata["best_epoch"],
    "best_mean_validation_RMSE": run_metadata["best_mean_validation_RMSE"],
    "final_training_metrics": run_metadata["final_training_metrics"],
    "validation_metrics": run_metadata["validation_metrics"],
    "test_evaluated": "NO",
})
